In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import psycopg2

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
print("Conexão ok")


Conexão ok


In [2]:
with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks (
            id SERIAL PRIMARY KEY,
            texto TEXT NOT NULL,
            embedding vector(1536),
            fonte TEXT
        );
    """)
    conn.commit()
print("Tabela criada")

Tabela criada


In [3]:
def gerar_embedding(texto: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texto
    )
    return response.data[0].embedding

# testa
emb = gerar_embedding("direitos fundamentais")
print(f"Embedding gerado: {len(emb)} dimensões")

Embedding gerado: 1536 dimensões


In [4]:
trechos = [
    ("Art. 1º A República Federativa do Brasil, formada pela união indissolúvel dos Estados e Municípios e do Distrito Federal, constitui-se em Estado Democrático de Direito.", "CF/1988 Art. 1º"),
    ("Art. 5º Todos são iguais perante a lei, sem distinção de qualquer natureza, garantindo-se aos brasileiros e aos estrangeiros residentes no País a inviolabilidade do direito à vida, à liberdade, à igualdade, à segurança e à propriedade.", "CF/1988 Art. 5º"),
    ("Art. 6º São direitos sociais a educação, a saúde, a alimentação, o trabalho, a moradia, o transporte, o lazer, a segurança, a previdência social, a proteção à maternidade e à infância, a assistência aos desamparados.", "CF/1988 Art. 6º"),
    ("Art. 196. A saúde é direito de todos e dever do Estado, garantido mediante políticas sociais e econômicas que visem à redução do risco de doença e de outros agravos.", "CF/1988 Art. 196"),
    ("Art. 205. A educação, direito de todos e dever do Estado e da família, será promovida e incentivada com a colaboração da sociedade, visando ao pleno desenvolvimento da pessoa.", "CF/1988 Art. 205"),
]

with conn.cursor() as cur:
    for texto, fonte in trechos:
        emb = gerar_embedding(texto)
        cur.execute(
            "INSERT INTO chunks (texto, embedding, fonte) VALUES (%s, %s, %s)",
            (texto, emb, fonte)
        )
    conn.commit()

print(f"{len(trechos)} chunks inseridos")

5 chunks inseridos


In [5]:
def buscar_chunks(pergunta: str, top_k: int = 3) -> list[dict]:
    emb = gerar_embedding(pergunta)
    with conn.cursor() as cur:
        cur.execute("""
            SELECT texto, fonte, embedding <-> %s::vector AS distancia
            FROM chunks
            ORDER BY distancia
            LIMIT %s
        """, (emb, top_k))
        rows = cur.fetchall()
    return [{"texto": r[0], "fonte": r[1], "distancia": r[2]} for r in rows]

# testa
resultados = buscar_chunks("quais são os direitos sociais?")
for r in resultados:
    print(f"[{r['fonte']}] distancia={r['distancia']:.4f}")
    print(r['texto'][:100])
    print()

[CF/1988 Art. 6º] distancia=0.7408
Art. 6º São direitos sociais a educação, a saúde, a alimentação, o trabalho, a moradia, o transporte

[CF/1988 Art. 196] distancia=0.9617
Art. 196. A saúde é direito de todos e dever do Estado, garantido mediante políticas sociais e econô

[CF/1988 Art. 205] distancia=0.9883
Art. 205. A educação, direito de todos e dever do Estado e da família, será promovida e incentivada 



In [6]:
def responder(pergunta: str) -> str:
    chunks = buscar_chunks(pergunta, top_k=3)
    
    contexto = "\n\n".join([
        f"Fonte: {c['fonte']}\n{c['texto']}"
        for c in chunks
    ])
    
    prompt = f"""Você é um assistente jurídico especializado na legislação brasileira.
Responda a pergunta abaixo usando APENAS as informações dos trechos fornecidos.
Se a resposta não estiver nos trechos, diga que não encontrou.

Trechos relevantes:
{contexto}

Pergunta: {pergunta}

Resposta:"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    
    resposta = response.choices[0].message.content
    
    print(f"Pergunta: {pergunta}")
    print(f"\nResposta: {resposta}")
    print(f"\nFontes usadas: {[c['fonte'] for c in chunks]}")
    
    return resposta

# testa
responder("Quais são os direitos sociais garantidos pela Constituição?")

Pergunta: Quais são os direitos sociais garantidos pela Constituição?

Resposta: Os direitos sociais garantidos pela Constituição são: a educação, a saúde, a alimentação, o trabalho, a moradia, o transporte, o lazer, a segurança, a previdência social, a proteção à maternidade e à infância, e a assistência aos desamparados.

Fontes usadas: ['CF/1988 Art. 6º', 'CF/1988 Art. 196', 'CF/1988 Art. 5º']


'Os direitos sociais garantidos pela Constituição são: a educação, a saúde, a alimentação, o trabalho, a moradia, o transporte, o lazer, a segurança, a previdência social, a proteção à maternidade e à infância, e a assistência aos desamparados.'

In [7]:
responder("O que é o Estado Democrático de Direito?")
responder("Qual é o direito à saúde previsto na Constituição?")
responder("Qual é a pena para homicídio?")  # não está nos chunks — deve dizer que não encontrou

Pergunta: O que é o Estado Democrático de Direito?

Resposta: O Estado Democrático de Direito é a República Federativa do Brasil, formada pela união indissolúvel dos Estados e Municípios e do Distrito Federal.

Fontes usadas: ['CF/1988 Art. 1º', 'CF/1988 Art. 205', 'CF/1988 Art. 196']
Pergunta: Qual é o direito à saúde previsto na Constituição?

Resposta: O direito à saúde previsto na Constituição é que a saúde é um direito de todos e dever do Estado, garantido mediante políticas sociais e econômicas que visem à redução do risco de doença e de outros agravos.

Fontes usadas: ['CF/1988 Art. 196', 'CF/1988 Art. 6º', 'CF/1988 Art. 205']
Pergunta: Qual é a pena para homicídio?

Resposta: Não encontrei.

Fontes usadas: ['CF/1988 Art. 5º', 'CF/1988 Art. 205', 'CF/1988 Art. 196']


'Não encontrei.'